In [ ]:
import os
import numpy as np
from IPython.display import display, HTML, Image
from PIL import Image as PILImage, ImageDraw, ImageFont
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import re
import warnings
import geopandas as gpd
from shapely.geometry import shape, Polygon, MultiPolygon, box, Point
import pandas as pd
from rasterio import features
from rasterio.transform import Affine
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import pickle
import sys
import matplotlib.patches as mpatches

# Suppress font warnings / Потискане на предупреждения за шрифтове
warnings.filterwarnings('ignore', category=UserWarning)

# ==================== CONFIGURATION ====================
# (English) Configuration of input/output directories
# (Bulgarian) Настройка на входни/изходни директории

# Path to the CSV file with fire suggestions (originally fires_suggestion.csv)
# Път към CSV файла с предложения за пожари (първоначално fires_suggestion.csv)
file_path = r'D:\data\master_thesis\input\fires_suggestion.csv'

# Directory where Sentinel-2 composite images are stored
# Директория, където се съхраняват композитните изображения от Sentinel-2
PREVIEW_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures'

# Output directory for the Random Forest classification results
# Изходна директория за резултатите от класификацията с Random Forest
CLASSIFIED_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_report\random_forest'

# Directory for vector exports (GeoJSON etc.) – created as a subdirectory of CLASSIFIED_DIR
# Директория за векторни експорти (GeoJSON и др.) – създава се като поддиректория на CLASSIFIED_DIR
VECTOR_EXPORT_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_report\random_forest\vector_exports'

# Directory containing training polygon files (GeoPackage) for each class
# Директория с тренировъчни полигони (GeoPackage) за всеки клас
POLYGON_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures\polygons'

# Directory with urban mask GeoPackages used to force urban pixels into class 3
# Директория с GeoPackage маски на урбанизирани територии, използвани за принудително задаване на клас 3
URBAN_MASK_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures\geopackages'

# Create directories if they don't exist / Създаване на директории, ако не съществуват
os.makedirs(CLASSIFIED_DIR, exist_ok=True)
os.makedirs(VECTOR_EXPORT_DIR, exist_ok=True)

# Define class names and colors / Дефиниране на имена и цветове на класовете
CLASSES = {
    1: {'name': 'Urban', 'color': (255, 0, 0), 'polygon_file': 'urban'},
    2: {'name': 'Water', 'color': (0, 0, 255), 'polygon_file': 'water'},
    3: {'name': 'Bare and Urban Territories', 'color': (139, 69, 19), 'polygon_file': 'bare_lands'},
    4: {'name': 'Field/Agriculture', 'color': (255, 255, 0), 'polygon_file': 'field_agriculture'},
    5: {'name': 'Coniferous Forest', 'color': (0, 100, 0), 'polygon_file': 'coniferous'},
    6: {'name': 'Deciduous Forest', 'color': (0, 255, 0), 'polygon_file': 'forest_deciduous'}
}

# ==================== CRS HANDLING ====================
# (English) Global cache for the CSV data to avoid repeated loading
# (Bulgarian) Глобален кеш за данните от CSV файла, за да се избегне многократно зареждане
_crs_cache = None

def _load_crs_csv():
    """
    (EN) Load the fires_suggestion.csv file once and cache it.
    (BG) Зарежда файла fires_suggestion.csv веднъж и го кешира.
    """
    global _crs_cache
    if _crs_cache is None:
        try:
            _crs_cache = pd.read_csv(file_path)
            print(f"✅ Loaded CRS information from {file_path} ({len(_crs_cache)} rows)")
        except Exception as e:
            print(f"❌ Could not load CRS file: {e}")
            _crs_cache = pd.DataFrame()  # empty to avoid repeated attempts
    return _crs_cache

def get_crs_for_fire(fire_num):
    """
    (EN) Retrieve the CRS string from fires_suggestion.csv for a given fire number.
         The CSV is expected to have a column that identifies the fire number (case-insensitive)
         and a column named 'crs'.
    (BG) Извлича CRS низа от fires_suggestion.csv за даден номер на пожар.
         Очаква се CSV файлът да има колона, идентифицираща номера на пожара (нечувствителна към регистъра),
         и колона с име 'crs'.
    """
    df = _load_crs_csv()
    if df.empty:
        return None

    # Case‑insensitive search for a fire ID column
    target_cols = ['fire_number', 'fire_num', 'fire_id', 'firenumber']
    # Build a mapping: lowercase column -> actual column name
    cols_lower = {col.lower(): col for col in df.columns}
    fire_col = None
    for candidate in target_cols:
        if candidate in cols_lower:
            fire_col = cols_lower[candidate]
            break

    if fire_col is None:
        # Try a looser match: any column containing 'fire' and 'id' or 'number'
        for col in df.columns:
            low = col.lower()
            if 'fire' in low and ('id' in low or 'number' in low):
                fire_col = col
                break

    if fire_col is None:
        print(f"⚠ Warning: Could not find a fire identifier column in CSV. Available columns: {df.columns.tolist()}")
        return None

    # Print which column is being used for clarity
    print(f"  🔍 Using column '{fire_col}' to match fire numbers")

    matches = df[df[fire_col] == fire_num]
    if matches.empty:
        print(f"⚠ Warning: Fire number {fire_num} not found in CSV ({fire_col} column)")
        return None

    crs_series = matches['crs'].dropna()
    if crs_series.empty:
        print(f"⚠ Warning: No CRS value found for fire {fire_num} in CSV")
        return None

    crs_value = crs_series.iloc[0]
    print(f"  📌 CRS from CSV for fire {fire_num}: {crs_value}")
    return str(crs_value)

def reproject_classification(classification, src_transform, src_crs, dst_crs):
    """
    (EN) Reproject a classification raster from source CRS to destination CRS.
         Returns the reprojected array and the new geotransform.
    (BG) Препроектира класификационен растер от изходен CRS към целеви CRS.
         Връща препроектирания масив и новия геотрансформ.
    """
    src_height, src_width = classification.shape
    # Calculate output dimensions and transform
    dst_transform, dst_width, dst_height = calculate_default_transform(
        src_crs, dst_crs, src_width, src_height,
        left=src_transform[2],
        bottom=src_transform[5] + src_height * src_transform[4],
        right=src_transform[2] + src_width * src_transform[0],
        top=src_transform[5]
    )
    # Prepare destination array
    dst_classification = np.zeros((dst_height, dst_width), dtype=classification.dtype)

    # Reproject using nearest neighbour (suitable for classification)
    reproject(
        source=classification,
        destination=dst_classification,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest
    )
    return dst_classification, dst_transform

# ==================== HELPER FUNCTIONS ====================
def extract_fire_number_from_filename(filename):
    """
    (EN) Extract fire number from a filename using regex.
    (BG) Извличане на номер на пожар от име на файл чрез регулярен израз.
    """
    fire_match = re.search(r'fire[_\s]?(\d+)', filename.lower())
    if fire_match:
        return int(fire_match.group(1))
    return None

def get_fire_id_formats(fire_num):
    """
    (EN) Return a dictionary with different string representations of a fire ID.
    (BG) Връща речник с различни низови представяния на идентификатор на пожар.
    """
    return {
        'fire_num': str(fire_num),
        'fire_01': f"fire_{fire_num:02d}",  # fire_01 format
        'fire_1': f"fire_{fire_num}",       # fire_1 format
        'fire1': f"fire{fire_num}"          # fire1 format
    }

def find_polygon_directory(fire_id_dict):
    """
    (EN) Find the polygon directory that matches the given fire ID (tries several naming conventions).
    (BG) Намира директорията с полигони, съответстваща на идентификатора на пожара (пробва различни конвенции за имена).
    """
    possible_dirs = [
        os.path.join(POLYGON_DIR, fire_id_dict['fire_01']),  # fire_01
        os.path.join(POLYGON_DIR, fire_id_dict['fire_1']),   # fire_1
        os.path.join(POLYGON_DIR, fire_id_dict['fire1'])     # fire1
    ]

    for dir_path in possible_dirs:
        if os.path.exists(dir_path):
            return dir_path

    # If directory doesn't exist, check if polygons are directly in POLYGON_DIR
    print(f"  - Checking for loose polygon files... / Проверка за разпръснати полигонови файлове...")
    return POLYGON_DIR  # Return base directory to check for loose files

def load_polygon_samples(fire_id_dict):
    """
    (EN) Load training polygon samples for a specific fire. Generates random points inside each polygon.
         Returns arrays of coordinates and corresponding class labels.
    (BG) Зарежда тренировъчни полигони за конкретен пожар. Генерира случайни точки във всеки полигон.
         Връща масиви от координати и съответните класови етикети.
    """
    fire_num = fire_id_dict['fire_num']

    # Find the correct polygon directory
    polygon_dir = find_polygon_directory(fire_id_dict)

    if polygon_dir == POLYGON_DIR:
        print(f"  - Looking for polygon files directly in {POLYGON_DIR}")
        # Files might be directly in the polygon directory
        all_points = []
        all_labels = []

        for class_id, class_info in CLASSES.items():
            # Try different filename patterns
            possible_files = [
                f"fire{fire_num}_{class_info['polygon_file']}.gpkg",  # fire1_water.gpkg
                f"fire_{fire_num}_{class_info['polygon_file']}.gpkg",  # fire_1_water.gpkg
                f"fire_{int(fire_num):02d}_{class_info['polygon_file']}.gpkg"  # fire_01_water.gpkg
            ]

            file_found = False
            for polygon_file in possible_files:
                polygon_path = os.path.join(polygon_dir, polygon_file)

                if os.path.exists(polygon_path):
                    try:
                        gdf = gpd.read_file(polygon_path)
                        if len(gdf) > 0:
                            # Sample points from polygons
                            print(f"  - Loading {polygon_file}...")
                            for geometry in gdf.geometry:
                                if geometry.geom_type == 'Polygon':
                                    # Generate random points within polygon
                                    minx, miny, maxx, maxy = geometry.bounds
                                    n_points = min(100, max(10, int(geometry.area / 10000)))  # Adaptive sampling
                                    points = []

                                    for _ in range(n_points):
                                        attempts = 0
                                        while attempts < 100:  # Safety limit
                                            point = Point(np.random.uniform(minx, maxx),
                                                         np.random.uniform(miny, maxy))
                                            if geometry.contains(point):
                                                points.append(point)
                                                break
                                            attempts += 1

                                    # Add points and labels - ensure class_id is integer
                                    for point in points:
                                        all_points.append((point.x, point.y))
                                        all_labels.append(int(class_id))  # Ensure integer

                            print(f"    ✓ Loaded {len(gdf)} {class_info['name']} polygons ({n_points} points each)")
                            file_found = True
                            break
                        else:
                            print(f"    ⚠ No features in {polygon_file}")
                    except Exception as e:
                        print(f"    ❌ Error loading {polygon_file}: {e}")

            if not file_found:
                print(f"  - {class_info['name']} polygons not found (skipping)")

        if not all_points:
            print(f"  ❌ No polygon files found for fire {fire_num}")
            return None, None

        return np.array(all_points), np.array(all_labels)

    else:
        # Directory exists, load from there
        print(f"  - Loading polygons from {polygon_dir}")
        all_points = []
        all_labels = []

        for class_id, class_info in CLASSES.items():
            # Try different filename patterns within the directory
            possible_files = [
                f"fire{fire_num}_{class_info['polygon_file']}.gpkg",
                f"fire_{fire_num}_{class_info['polygon_file']}.gpkg",
                f"{fire_num}_{class_info['polygon_file']}.gpkg",
                f"{class_info['polygon_file']}.gpkg"
            ]

            file_found = False
            for polygon_file in possible_files:
                polygon_path = os.path.join(polygon_dir, polygon_file)

                if os.path.exists(polygon_path):
                    try:
                        gdf = gpd.read_file(polygon_path)
                        if len(gdf) > 0:
                            # Sample points from polygons
                            print(f"  - Loading {polygon_file}...")
                            for geometry in gdf.geometry:
                                if geometry.geom_type == 'Polygon':
                                    # Generate random points within polygon
                                    minx, miny, maxx, maxy = geometry.bounds
                                    n_points = min(100, max(10, int(geometry.area / 10000)))
                                    points = []

                                    for _ in range(n_points):
                                        attempts = 0
                                        while attempts < 100:
                                            point = Point(np.random.uniform(minx, maxx),
                                                         np.random.uniform(miny, maxy))
                                            if geometry.contains(point):
                                                points.append(point)
                                                break
                                            attempts += 1

                                    # Add points and labels - ensure class_id is integer
                                    for point in points:
                                        all_points.append((point.x, point.y))
                                        all_labels.append(int(class_id))  # Ensure integer

                            print(f"    ✓ Loaded {len(gdf)} {class_info['name']} polygons")
                            file_found = True
                            break
                    except Exception as e:
                        print(f"    ❌ Error loading {polygon_file}: {e}")

            if not file_found:
                print(f"  - {class_info['name']} polygons not found (skipping)")

        if not all_points:
            print(f"  ❌ No valid polygons found in directory")
            return None, None

        return np.array(all_points), np.array(all_labels)

def load_urban_mask(fire_id_dict):
    """
    (EN) Load urban mask GeoPackage for a fire. The mask is used to overwrite predictions in urban areas.
    (BG) Зарежда GeoPackage маска на урбанизирани територии за пожар. Маската се използва за презаписване на прогнози в урбанизирани зони.
    """
    fire_num = fire_id_dict['fire_num']

    # Try different filename patterns
    possible_files = [
        f"fire{fire_num}_urban_masks.gpkg",      # fire1_urban_masks.gpkg
        f"fire_{fire_num}_urban_masks.gpkg",     # fire_1_urban_masks.gpkg
        f"fire_{int(fire_num):02d}_urban_masks.gpkg",  # fire_01_urban_masks.gpkg
        f"fire{fire_num}_urban_mask.gpkg",       # fire1_urban_mask.gpkg
        f"fire_{fire_num}_urban_mask.gpkg",      # fire_1_urban_mask.gpkg
        f"urban_mask_{fire_num}.gpkg",           # urban_mask_1.gpkg
        f"urban_masks_fire{fire_num}.gpkg",      # urban_masks_fire1.gpkg
        f"urban_masks.gpkg"                      # generic urban_masks.gpkg
    ]

    # Debug: list available files
    print(f"  - Looking for urban masks for fire {fire_num} in {URBAN_MASK_DIR}")
    if os.path.exists(URBAN_MASK_DIR):
        available_files = os.listdir(URBAN_MASK_DIR)
        urban_files = [f for f in available_files if 'urban' in f.lower() and f.endswith('.gpkg')]
        if urban_files:
            print(f"  - Available urban mask files: {urban_files}")
        else:
            print(f"  - No urban mask files found in directory")

    for urban_file in possible_files:
        urban_mask_path = os.path.join(URBAN_MASK_DIR, urban_file)

        if os.path.exists(urban_mask_path):
            try:
                gdf = gpd.read_file(urban_mask_path)
                if len(gdf) > 0:
                    print(f"  ✓ Urban mask loaded: {urban_file} ({len(gdf)} polygons)")

                    # Check CRS
                    if gdf.crs is None:
                        print(f"  ⚠ Urban mask has no CRS assigned, assuming WGS84")
                        gdf = gdf.set_crs('EPSG:4326', allow_override=True)

                    # Calculate total area
                    total_area = sum(gdf.geometry.area)
                    print(f"    Total urban area: {total_area:.2f} square degrees")

                    return gdf
                else:
                    print(f"  ⚠ Urban mask file is empty: {urban_file}")
            except Exception as e:
                print(f"  ❌ Error loading urban mask {urban_file}: {e}")

    print(f"  ⚠ No urban mask found for fire {fire_num}")
    return None

def extract_band_values_at_points(raster_path, points):
    """
    (EN) Extract spectral band values at given geographic points from a raster.
    (BG) Извлича стойностите на спектралните канали в зададени географски точки от растер.
    """
    with rasterio.open(raster_path) as src:
        band_values = []
        valid_indices = []

        print(f"    Extracting values for {len(points)} points...")

        for i, (x, y) in enumerate(points):
            # Transform point to pixel coordinates
            row, col = src.index(x, y)

            # Check if point is within raster bounds
            if 0 <= row < src.height and 0 <= col < src.width:
                # Read all band values at this pixel
                pixel_values = []
                for band_idx in range(1, src.count + 1):
                    band_data = src.read(band_idx)
                    pixel_values.append(band_data[row, col])

                band_values.append(pixel_values)
                valid_indices.append(i)

        print(f"    ✓ Valid points within raster: {len(valid_indices)}/{len(points)}")
        return np.array(band_values), np.array(valid_indices)  # Ensure numpy array

def identify_sentinel2_bands_correctly(bands, src_descriptions=None):
    """
    (EN) Automatically identify Sentinel-2 bands (Blue, Green, Red, NIR) based on descriptions or reflectance.
    (BG) Автоматично идентифицира каналите на Sentinel-2 (Син, Зелен, Червен, БИЧ) по описания или отражателна способност.
    """
    print(f"  - Number of bands: {len(bands)}")

    if src_descriptions:
        print(f"  - Band descriptions: {src_descriptions}")

    if src_descriptions:
        band_mapping = {}
        for i, desc in enumerate(src_descriptions):
            if not desc:
                continue
            desc = desc.upper()
            if 'B02' in desc or desc == 'B2':
                band_mapping['blue'] = bands[i]
                print(f"    Found Blue (B02) at band {i+1}")
            elif 'B03' in desc or desc == 'B3':
                band_mapping['green'] = bands[i]
                print(f"    Found Green (B03) at band {i+1}")
            elif 'B04' in desc or desc == 'B4':
                band_mapping['red'] = bands[i]
                print(f"    Found Red (B04) at band {i+1}")
            elif 'B08' in desc or desc == 'B8':
                band_mapping['nir'] = bands[i]
                print(f"    Found NIR (B08) at band {i+1}")
            elif 'B8A' in desc:
                if 'nir' not in band_mapping:
                    band_mapping['nir'] = bands[i]
                    print(f"    Found NIR (B8A) at band {i+1}")

        if len(band_mapping) == 4:
            print("  - Successfully identified bands from descriptions")
            return band_mapping

    if len(bands) >= 15:
        print("  - Detected Sentinel-2 L2A product (15 bands)")
        band_info = {
            'blue': bands[2],
            'green': bands[3],
            'red': bands[4],
            'nir': bands[8]
        }
        return band_info

    if len(bands) >= 12:
        band_info = {
            'blue': bands[1],
            'green': bands[2],
            'red': bands[3],
            'nir': bands[7]
        }
        print("  - Identified as 12-band Sentinel-2")
        return band_info

    elif len(bands) >= 4:
        band_means = [np.mean(band) for band in bands[:4]]
        nir_idx = np.argmax(band_means)
        red_idx = np.argmin(band_means[:3])
        remaining = [i for i in range(4) if i not in [nir_idx, red_idx]]
        blue_idx = remaining[0]
        green_idx = remaining[1]

        band_info = {
            'blue': bands[blue_idx],
            'green': bands[green_idx],
            'red': bands[red_idx],
            'nir': bands[nir_idx]
        }
        print(f"  - Identified by reflectance")
        return band_info

    else:
        band_info = {
            'blue': bands[0] if len(bands) > 0 else None,
            'green': bands[1] if len(bands) > 1 else None,
            'red': bands[2] if len(bands) > 2 else None,
            'nir': bands[3] if len(bands) > 3 else None
        }
        print("  - Using standard band order assumption")
        return band_info

def calculate_indices(band_info):
    """
    (EN) Calculate NDVI and NDWI from the identified bands.
    (BG) Изчислява NDVI и NDWI от идентифицираните канали.
    """
    indices = {}

    # NDVI
    if 'red' in band_info and 'nir' in band_info:
        red = band_info['red'].astype(np.float32)
        nir = band_info['nir'].astype(np.float32)
        denominator = nir + red
        valid_mask = denominator > 0
        ndvi = np.zeros_like(red, dtype=np.float32)
        ndvi[valid_mask] = (nir[valid_mask] - red[valid_mask]) / denominator[valid_mask]
        indices['ndvi'] = np.clip(ndvi, -1, 1)

    # NDWI
    if 'green' in band_info and 'nir' in band_info:
        green = band_info['green'].astype(np.float32)
        nir = band_info['nir'].astype(np.float32)
        denominator = green + nir
        valid_mask = denominator > 0
        ndwi = np.zeros_like(green, dtype=np.float32)
        ndwi[valid_mask] = (green[valid_mask] - nir[valid_mask]) / denominator[valid_mask]
        indices['ndwi'] = np.clip(ndwi, -1, 1)

    return indices

def create_true_color_rgb(band_info):
    """
    (EN) Create a true‑colour RGB image from the Red, Green, Blue bands.
    (BG) Създава истинско цветно RGB изображение от червения, зеления и синия канал.
    """
    if 'red' in band_info and 'green' in band_info and 'blue' in band_info:
        red = band_info['red']
        green = band_info['green']
        blue = band_info['blue']

        # Enhance bands for better visualization
        def enhance_band(band):
            p2 = np.percentile(band, 2)
            p98 = np.percentile(band, 98)
            enhanced = np.clip((band - p2) / (p98 - p2), 0, 1)
            return (enhanced * 255).astype(np.uint8)

        red_enhanced = enhance_band(red)
        green_enhanced = enhance_band(green)
        blue_enhanced = enhance_band(blue)

        return np.dstack((red_enhanced, green_enhanced, blue_enhanced))
    else:
        print("❌ Missing bands for true color RGB")
        return None

def train_random_forest_classifier(raster_path, training_points, training_labels):
    """
    (EN) Train a Random Forest classifier using spectral bands (and indices) at training points.
         Includes validation, handling of class imbalance, and feature importance reporting.
    (BG) Обучава класификатор Random Forest, използвайки спектрални канали (и индекси) в тренировъчните точки.
         Включва валидация, справяне с небалансирани класове и отчитане на важността на характеристиките.
    """
    print("  - Extracting band values at training points...")
    X, valid_indices = extract_band_values_at_points(raster_path, training_points)

    if len(X) == 0:
        print("  ❌ No valid training points found within raster bounds")
        return None

    # Filter labels to match valid indices
    y = training_labels[valid_indices]

    # Validate and clean labels
    print("  - Validating class labels...")

    # Convert labels to integer and filter out invalid values
    y_clean = []
    valid_label_indices = []

    for i, label in enumerate(y):
        try:
            # Try to convert to integer
            label_int = int(float(label))
            # Check if it's a valid class ID
            if label_int in CLASSES:
                y_clean.append(label_int)
                valid_label_indices.append(i)
            else:
                print(f"    ⚠ Skipping invalid class ID: {label}")
        except (ValueError, TypeError):
            print(f"    ⚠ Skipping non-numeric label: {label}")

    if len(y_clean) == 0:
        print("  ❌ No valid class labels found")
        return None

    # Filter X to match valid labels
    X_clean = X[valid_label_indices]
    y_clean = np.array(y_clean)

    print(f"  - Cleaned labels: {len(y_clean)} samples")

    # Check class distribution
    unique_classes, class_counts = np.unique(y_clean, return_counts=True)
    print(f"\n  - Class distribution (cleaned):")
    for class_id, count in zip(unique_classes, class_counts):
        class_name = CLASSES.get(class_id, {}).get('name', f'Class_{class_id}')
        print(f"    {class_name}: {count} samples")

    # Check if we have enough samples for each class
    min_samples_per_class = 2
    classes_with_enough_samples = []
    for class_id, count in zip(unique_classes, class_counts):
        if count >= min_samples_per_class:
            classes_with_enough_samples.append(class_id)

    if len(classes_with_enough_samples) < 2:
        print(f"  ❌ Not enough classes with sufficient samples (need at least 2 classes with {min_samples_per_class}+ samples each)")
        return None

    # Filter to only include classes with enough samples
    mask = np.isin(y_clean, classes_with_enough_samples)
    X_filtered = X_clean[mask]
    y_filtered = y_clean[mask]

    print(f"  - Training with {len(classes_with_enough_samples)} classes that have sufficient samples")

    # Open raster again to read bands and calculate indices
    with rasterio.open(raster_path) as src:
        bands = []
        for i in range(1, src.count + 1):
            bands.append(src.read(i))

        band_info = identify_sentinel2_bands_correctly(bands, src.descriptions if hasattr(src, 'descriptions') and src.descriptions else None)
        indices = calculate_indices(band_info)

    # Add indices to features if available
    if 'ndvi' in indices or 'ndwi' in indices:
        print("  - Adding spectral indices to features...")
        X_with_indices = []

        # Get the indices for the filtered points
        # First get indices of valid points after all filters
        filtered_valid_indices = valid_indices[valid_label_indices][mask]
        valid_points = training_points[filtered_valid_indices]

        # Reopen raster to get pixel coordinates
        with rasterio.open(raster_path) as src:
            for i, (x, y) in enumerate(valid_points):
                row, col = src.index(x, y)
                if 0 <= row < src.height and 0 <= col < src.width:
                    features_list = X_filtered[i].tolist()
                    if 'ndvi' in indices:
                        features_list.append(indices['ndvi'][row, col])
                    if 'ndwi' in indices:
                        features_list.append(indices['ndwi'][row, col])
                    X_with_indices.append(features_list)

        X_filtered = np.array(X_with_indices)

    # Train-test split with stratification
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X_filtered, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
        )
    except ValueError as e:
        print(f"  ⚠ Could not stratify: {e}")
        print("  - Using random split instead...")
        X_train, X_test, y_train, y_test = train_test_split(
            X_filtered, y_filtered, test_size=0.2, random_state=42
        )

    print(f"  - Training samples: {len(X_train)}")
    print(f"  - Testing samples: {len(X_test)}")

    # Train Random Forest
    print("  - Training Random Forest classifier...")
    rf_classifier = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'  # Handle imbalanced classes
    )

    rf_classifier.fit(X_train, y_train)

    # Evaluate on test set
    y_pred = rf_classifier.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print(f"\n  - Random Forest accuracy: {accuracy:.3f}")
    print("\n  - Classification Report:")

    # Get class names for the report
    target_names = []
    for class_id in sorted(np.unique(y_filtered)):
        target_names.append(CLASSES.get(class_id, {}).get('name', f'Class_{class_id}'))

    print(classification_report(y_test, y_pred, target_names=target_names))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print("\n  - Confusion Matrix:")
    print(cm)

    # Feature importance
    feature_names = [f"Band_{i+1}" for i in range(len(X_filtered[0]) - (1 if 'ndvi' in indices else 0) - (1 if 'ndwi' in indices else 0))]
    if 'ndvi' in indices:
        feature_names.append('NDVI')
    if 'ndwi' in indices:
        feature_names.append('NDWI')

    print("\n  - Top 10 Feature Importances:")
    importances = rf_classifier.feature_importances_
    sorted_idx = np.argsort(importances)[::-1][:10]

    for idx in sorted_idx:
        print(f"    {feature_names[idx] if idx < len(feature_names) else f'Feature_{idx}'}: {importances[idx]:.4f}")

    return rf_classifier

def classify_raster_with_rf(raster_path, classifier, urban_mask=None):
    """
    (EN) Apply the trained Random Forest to the entire raster. If an urban mask is provided,
         force urban areas to class 3 (Bare and Urban Territories).
    (BG) Прилага обучения Random Forest върху целия растер. Ако е предоставена урбан маска,
         принудително задава урбанизираните територии към клас 3 (Голи и урбанизирани територии).
    """
    print("  - Classifying raster...")

    with rasterio.open(raster_path) as src:
        # Read all bands
        bands = []
        for i in range(1, src.count + 1):
            bands.append(src.read(i))

        height, width = bands[0].shape
        print(f"  - Raster dimensions: {height} x {width}")

        band_info = identify_sentinel2_bands_correctly(bands, src.descriptions if hasattr(src, 'descriptions') and src.descriptions else None)
        indices = calculate_indices(band_info)

        # Prepare features for classification
        print("  - Preparing features for classification...")

        # Reshape bands for classification
        band_arrays = []
        for band in bands:
            band_arrays.append(band.reshape(-1, 1))

        X = np.hstack(band_arrays)

        # Add indices if available
        if 'ndvi' in indices:
            ndvi_flat = indices['ndvi'].reshape(-1, 1)
            X = np.hstack([X, ndvi_flat])

        if 'ndwi' in indices:
            ndwi_flat = indices['ndwi'].reshape(-1, 1)
            X = np.hstack([X, ndwi_flat])

        # Classify in batches to manage memory
        print("  - Running Random Forest classification...")
        batch_size = 100000
        y_pred = np.zeros(height * width, dtype=np.uint8)

        for i in range(0, len(X), batch_size):
            batch_end = min(i + batch_size, len(X))
            y_pred[i:batch_end] = classifier.predict(X[i:batch_end])
            progress = (batch_end / len(X)) * 100
            print(f"    Progress: {progress:.1f}%", end='\r')

        print(f"    Progress: 100.0%")

        # Reshape back to image
        classification = y_pred.reshape(height, width)

        # BEFORE URBAN MASK: Count Bare and Urban Territories pixels
        bare_urban_before = np.sum(classification == 3)
        print(f"    Bare and Urban Territories BEFORE urban mask: {bare_urban_before:,} pixels")

        # Apply urban mask if available
        if urban_mask is not None:
            print("  - Applying urban mask...")

            # Ensure urban mask is in same CRS as raster
            if urban_mask.crs != src.crs:
                print(f"    Transforming urban mask from {urban_mask.crs} to {src.crs}")
                urban_mask = urban_mask.to_crs(src.crs)

            # Create mask raster
            urban_mask_raster = np.zeros((height, width), dtype=bool)

            for geometry in urban_mask.geometry:
                if geometry.geom_type == 'Polygon' or geometry.geom_type == 'MultiPolygon':
                    # Rasterize urban area
                    shapes = [(geometry, 1)]
                    urban_rasterized = features.rasterize(
                        shapes,
                        out_shape=(height, width),
                        transform=src.transform,
                        fill=0,
                        dtype=np.uint8
                    )
                    urban_mask_raster = urban_mask_raster | (urban_rasterized == 1)

            # Count urban mask pixels
            urban_pixels = np.sum(urban_mask_raster)
            print(f"    Urban mask covers: {urban_pixels:,} pixels")

            # Check what classes are being overwritten
            classes_in_urban_area = classification[urban_mask_raster]
            unique_classes, class_counts = np.unique(classes_in_urban_area, return_counts=True)
            print(f"    Classes in urban mask area BEFORE applying:")
            for class_id, count in zip(unique_classes, class_counts):
                class_name = CLASSES.get(class_id, {}).get('name', f'Class_{class_id}')
                print(f"      {class_name} (class {class_id}): {count:,} pixels")

            # Set urban areas to class 3 (Bare and Urban Territories)
            classification[urban_mask_raster] = 3

            # AFTER URBAN MASK: Count Bare and Urban Territories pixels
            bare_urban_after = np.sum(classification == 3)
            added_by_mask = bare_urban_after - bare_urban_before
            print(f"    Bare and Urban Territories AFTER urban mask: {bare_urban_after:,} pixels")
            print(f"    ✓ Applied urban mask to {urban_pixels:,} pixels (assigned to Bare and Urban Territories)")
            print(f"    ✓ Added {added_by_mask:,} pixels to Bare and Urban Territories class")
        else:
            print("  - No urban mask available for this fire")

        return classification, src.transform, src.crs, band_info

def create_classification_visualization(classification):
    """
    (EN) Create an RGB image where each class is coloured according to the predefined colour map.
    (BG) Създава RGB изображение, където всеки клас е оцветен според предварително зададената цветова схема.
    """
    height, width = classification.shape
    vis_rgb = np.zeros((height, width, 3), dtype=np.uint8)

    for class_id, class_info in CLASSES.items():
        mask = classification == class_id
        vis_rgb[mask] = class_info['color']

    return vis_rgb

def export_classification_results(classification, transform, crs, fire_id, output_dir):
    """
    (EN) Export the classification as a GeoTIFF and as vector polygons (GeoJSON).
         The CRS provided is used for both outputs.
    (BG) Експортира класификацията като GeoTIFF и като векторни полигони (GeoJSON).
         Предоставеният CRS се използва и за двата изхода.
    """
    print("  - Exporting classification results...")

    # Create output filename
    output_base = f"{fire_id}_random_forest_classification"

    # Export as GeoTIFF
    raster_path = os.path.join(output_dir, f"{output_base}.tif")
    with rasterio.open(
        raster_path,
        'w',
        driver='GTiff',
        height=classification.shape[0],
        width=classification.shape[1],
        count=1,
        dtype=classification.dtype,
        crs=crs,
        transform=transform
    ) as dst:
        dst.write(classification, 1)
        # Create colormap
        colormap = {}
        for class_id, class_info in CLASSES.items():
            colormap[class_id] = class_info['color']
        dst.write_colormap(1, colormap)

    print(f"    ✅ Raster saved: {raster_path}")

    # Export to vector format
    vector_dir = os.path.join(output_dir, "vector_exports")
    os.makedirs(vector_dir, exist_ok=True)

    geojson_path = os.path.join(vector_dir, f"{output_base}.geojson")

    print("  - Polygonizing classification...")
    # Polygonize
    shapes = features.shapes(classification.astype(np.int16), transform=transform)

    geometries = []
    properties = []

    for shape_dict, value in shapes:
        if value > 0:  # Skip background
            geometry = shape(shape_dict)
            if not geometry.is_empty and geometry.area > 100:  # Minimum area threshold
                geometries.append(geometry)
                class_name = CLASSES.get(int(value), {}).get('name', f'Class_{value}')
                area_sq_m = int(round(geometry.area * (abs(transform[0]) ** 2)))

                properties.append({
                    'class_id': int(value),
                    'class_name': class_name,
                    'area_sqm': area_sq_m,
                    'area_ha': area_sq_m / 10000.0
                })

    if geometries:
        gdf = gpd.GeoDataFrame(properties, geometry=geometries, crs=crs)
        gdf.to_file(geojson_path, driver='GeoJSON')
        print(f"    ✅ Vector export saved: {geojson_path}")
        print(f"    ✓ Created {len(geometries)} polygons")
    else:
        print("    ⚠ No valid polygons created")

    return raster_path, geojson_path

def create_classification_report(classification, band_info, fire_id, output_dir):
    """
    (EN) Generate a comprehensive report with statistics, charts, and the true-colour image.
    (BG) Генерира подробен доклад със статистики, графики и истинско цветно изображение.
    """
    print("  - Generating classification report...")

    # Calculate statistics
    total_pixels = classification.size
    stats = {}

    for class_id in np.unique(classification):
        if class_id > 0:  # Skip background
            class_pixels = np.sum(classification == class_id)
            percentage = (class_pixels / total_pixels) * 100
            class_name = CLASSES.get(class_id, {}).get('name', f'Class_{class_id}')

            stats[class_id] = {
                'name': class_name,
                'pixels': class_pixels,
                'percentage': percentage,
                'color': CLASSES.get(class_id, {}).get('color', (0, 0, 0))
            }

    # Create TRUE COLOR RGB visualization
    true_color_rgb = create_true_color_rgb(band_info)

    # Create classification visualization
    classification_vis = create_classification_visualization(classification)

    # Create visualization with 3x2 grid (cleaner layout without comparison)
    fig, axes = plt.subplots(3, 2, figsize=(16, 18))
    fig.suptitle(f'Random Forest Classification - {fire_id}', fontsize=20, fontweight='bold', y=0.98)

    # Plot 1: TRUE COLOR RGB (top-left)
    ax1 = axes[0, 0]
    if true_color_rgb is not None:
        ax1.imshow(true_color_rgb)
        ax1.set_title('TRUE COLOR RGB (Input Image)', fontsize=14, fontweight='bold', pad=15)
    else:
        ax1.text(0.5, 0.5, 'RGB Image\nNot Available', ha='center', va='center', fontsize=12)
        ax1.set_title('TRUE COLOR RGB', fontsize=14, fontweight='bold', pad=15)
    ax1.axis('off')

    # Plot 2: Classification map (top-right)
    ax2 = axes[0, 1]
    ax2.imshow(classification_vis)
    ax2.set_title('Random Forest Classification', fontsize=14, fontweight='bold', pad=15)
    ax2.axis('off')

    # Plot 3: Pie chart (middle-left) - make it larger
    ax3 = axes[1, 0]
    if stats:
        labels = [stats[class_id]['name'] for class_id in sorted(stats.keys())]
        sizes = [stats[class_id]['percentage'] for class_id in sorted(stats.keys())]
        colors = [tuple(c/255 for c in stats[class_id]['color']) for class_id in sorted(stats.keys())]

        # Create pie chart with better formatting
        wedges, texts, autotexts = ax3.pie(sizes, labels=labels, colors=colors,
                                          autopct='%1.1f%%', startangle=90,
                                          textprops={'fontsize': 10})

        # Improve autopct appearance
        for autotext in autotexts:
            autotext.set_color('white')
            autotext.set_fontweight('bold')
            autotext.set_fontsize(9)

        ax3.set_title('Land Cover Distribution', fontsize=14, fontweight='bold', pad=15)
        ax3.axis('equal')

    # Plot 4: Bar chart (middle-right)
    ax4 = axes[1, 1]
    if stats:
        class_ids = sorted(stats.keys())
        class_names = [stats[class_id]['name'] for class_id in class_ids]
        percentages = [stats[class_id]['percentage'] for class_id in class_ids]
        colors = [tuple(c/255 for c in stats[class_id]['color']) for class_id in class_ids]

        bars = ax4.bar(range(len(class_names)), percentages, color=colors, edgecolor='black', alpha=0.8)
        ax4.set_xlabel('Land Cover Class', fontsize=12, fontweight='bold')
        ax4.set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
        ax4.set_title('Land Cover Percentage', fontsize=14, fontweight='bold', pad=15)
        ax4.set_xticks(range(len(class_names)))
        ax4.set_xticklabels(class_names, rotation=45, ha='right', fontsize=10)
        ax4.tick_params(axis='y', labelsize=10)
        ax4.grid(True, alpha=0.3, linestyle='--')

        # Add value labels on bars with better formatting
        for bar, percentage in zip(bars, percentages):
            height = bar.get_height()
            ax4.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{percentage:.1f}%', ha='center', va='bottom',
                    fontsize=9, fontweight='bold')

    # Plot 5: Statistics table (bottom-left) - expanded
    ax5 = axes[2, 0]
    ax5.axis('off')

    if stats:
        table_data = []
        total_area = 0
        total_pixels_sum = 0

        for class_id in sorted(stats.keys()):
            stat = stats[class_id]
            area_ha = stat['pixels'] * 100 / 10000  # Approximate area in hectares
            total_area += area_ha
            total_pixels_sum += stat['pixels']
            table_data.append([
                stat['name'],
                f"{stat['pixels']:,}",
                f"{stat['percentage']:.2f}%",
                f"{area_ha:.1f}"
            ])

        # Add total row
        table_data.append([
            'TOTAL',
            f"{total_pixels_sum:,}",
            '100.00%',
            f"{total_area:.1f}"
        ])

        # Create table with better formatting
        table = ax5.table(
            cellText=table_data,
            colLabels=['Land Cover Class', 'Pixels', 'Percentage', 'Area (ha)'],
            cellLoc='center',
            loc='center',
            colWidths=[0.4, 0.2, 0.2, 0.2]
        )
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1.2, 1.8)

        # Style the table
        for (row, col), cell in table.get_celld().items():
            if row == 0:  # Header row
                cell.set_text_props(fontweight='bold', color='white')
                cell.set_facecolor('#2E86C1')
            elif row == len(table_data):  # Total row
                cell.set_text_props(fontweight='bold')
                cell.set_facecolor('#F0F0F0')

        ax5.set_title('Detailed Statistics', fontsize=14, fontweight='bold', pad=15)

    # Plot 6: Legend and summary (bottom-right)
    ax6 = axes[2, 1]
    ax6.axis('off')

    # Create legend with class colors
    legend_elements = []
    for class_id in sorted(stats.keys()):
        class_name = stats[class_id]['name']
        class_color = tuple(c/255 for c in stats[class_id]['color'])

        legend_elements.append(mpatches.Patch(
            color=class_color,
            label=class_name
        ))

    # Add legend
    if legend_elements:
        legend = ax6.legend(handles=legend_elements, loc='upper center',
                           fontsize=10, title="Land Cover Classes",
                           title_fontsize=12, ncol=2)
        legend.get_frame().set_alpha(0.9)

    # Calculate additional statistics
    if stats:
        # Find dominant class
        dominant_class = max(stats.items(), key=lambda x: x[1]['percentage'])

        # Calculate forest coverage
        forest_classes = [5, 6]  # Coniferous and Deciduous Forest
        forest_percentage = sum(stats[class_id]['percentage'] for class_id in forest_classes if class_id in stats)

        # Calculate vegetation coverage (Fields + Forests)
        vegetation_classes = [4, 5, 6]  # Field/Agriculture, Coniferous, Deciduous
        vegetation_percentage = sum(stats[class_id]['percentage'] for class_id in vegetation_classes if class_id in stats)

        # Calculate Bare and Urban Territories (including urban masks)
        bare_urban_classes = [3]  # Bare and Urban Territories
        bare_urban_percentage = sum(stats[class_id]['percentage'] for class_id in bare_urban_classes if class_id in stats)

        summary_text = [
            "CLASSIFICATION SUMMARY:",
            f"Total area analyzed: {total_area:.1f} ha",
            f"Dominant land cover: {dominant_class[1]['name']} ({dominant_class[1]['percentage']:.1f}%)",
            "",
            "COVERAGE ANALYSIS:",
            f"Forest Coverage: {forest_percentage:.1f}%",
            f"Vegetation (Fields + Forests): {vegetation_percentage:.1f}%",
            f"Bare & Urban Territories: {bare_urban_percentage:.1f}%",
            f"Water Bodies: {stats.get(2, {}).get('percentage', 0):.1f}%",
            "",
            "URBAN MASK INTEGRATION:",
            "• Urban mask areas are forced to class 3 (Bare and Urban Territories)",
            "• This ensures consistent urban area classification",
            "• Overrides Random Forest predictions in urban areas"
        ]

        ax6.text(0.02, 0.4, "\n".join(summary_text), transform=ax6.transAxes,
                fontsize=10, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle="round,pad=0.8", facecolor="lightyellow", alpha=0.7, edgecolor='gray'))

    plt.tight_layout()

    # Save report
    report_path = os.path.join(output_dir, f"{fire_id}_classification_report.png")
    plt.savefig(report_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

    print(f"    ✅ Report saved: {report_path}")

    # Save statistics to CSV
    csv_path = os.path.join(output_dir, f"{fire_id}_classification_statistics.csv")
    stats_df = pd.DataFrame([
        {
            'class_id': class_id,
            'class_name': stat['name'],
            'pixels': stat['pixels'],
            'percentage': stat['percentage'],
            'area_ha': stat['pixels'] * 100 / 10000
        }
        for class_id, stat in stats.items()
    ])
    stats_df.to_csv(csv_path, index=False)
    print(f"    ✅ Statistics saved: {csv_path}")

    return report_path, csv_path

def display_classification_report(report_path, fire_id):
    """
    (EN) Display the classification report PNG in a Jupyter notebook.
    (BG) Показва PNG доклада от класификацията в Jupyter среда.
    """
    print(f"\n📊 DISPLAYING CLASSIFICATION REPORT FOR {fire_id}")
    print("="*60)

    if os.path.exists(report_path):
        print(f"📈 Displaying classification report: {os.path.basename(report_path)}")

        # Display using IPython's Image display (cleaner for notebooks)
        display(HTML(f"<h2 style='color: #2E86C1;'>📊 Classification Report - {fire_id}</h2>"))
        display(HTML(f"<p><strong>Report file:</strong> {os.path.basename(report_path)}</p>"))
        display(Image(filename=report_path, width=1200))

        # Also show file path for reference
        print(f"\n📁 Report saved at: {report_path}")
    else:
        print(f"⚠ Report file not found: {report_path}")

    print("="*60)

def process_fire_by_number(fire_num):
    """
    (EN) Process a single fire: load training data, train Random Forest, classify, export results.
         The CRS from fires_suggestion.csv is used as the output CRS.
         If different from the raster CRS, the classification is reprojected.
    (BG) Обработва един пожар: зарежда тренировъчни данни, обучава Random Forest, класифицира, експортира резултати.
         CRS от fires_suggestion.csv се използва като изходен CRS.
         Ако е различен от CRS на растера, класификацията се препроектира.
    """
    print(f"\n{'='*60}")
    print(f"🔥 PROCESSING FIRE {fire_num:02d}")
    print(f"{'='*60}")

    # Get fire ID formats
    fire_id_dict = get_fire_id_formats(fire_num)
    display_fire_id = fire_id_dict['fire_01']

    # Find the corresponding image file
    image_file = None
    possible_patterns = [
        f"fire{fire_num}_simple_average_all_bands_2024.tif",
        f"fire{fire_num}_simple_average_all_bands_2024.tiff",
        f"fire_{fire_num}_simple_average_all_bands_2024.tif",
        f"fire_{fire_num}_simple_average_all_bands_2024.tiff",
        f"fire{fire_num}_simple_average.tif",
        f"fire_{fire_num}_simple_average.tif"
    ]

    for pattern in possible_patterns:
        file_path = os.path.join(PREVIEW_DIR, pattern)
        if os.path.exists(file_path):
            image_file = pattern
            break

    if not image_file:
        print(f"❌ No image file found for fire {fire_num}")
        return None

    img_path = os.path.join(PREVIEW_DIR, image_file)

    try:
        # Check if file exists
        if not os.path.exists(img_path):
            print(f"❌ File not found: {img_path}")
            return None

        # Load training polygons
        print("📚 Loading training polygons...")
        training_points, training_labels = load_polygon_samples(fire_id_dict)

        if training_points is None or len(training_points) == 0:
            print(f"❌ No training samples found for fire {fire_num}")
            return None

        print(f"  ✓ Total training samples: {len(training_points)}")

        # Load urban mask
        print("🏙️  Loading urban mask...")
        urban_mask = load_urban_mask(fire_id_dict)

        # Train Random Forest classifier
        print("🌲 Training Random Forest classifier...")
        rf_classifier = train_random_forest_classifier(img_path, training_points, training_labels)

        if rf_classifier is None:
            print("❌ Failed to train classifier")
            return None

        # Classify the entire raster
        print("🖼️  Classifying raster...")
        classification, transform, raster_crs, band_info = classify_raster_with_rf(img_path, rf_classifier, urban_mask)

        # ------------------------------------------------------------
        # CRS handling: always use the CSV CRS (with fallback)
        # ------------------------------------------------------------
        csv_crs = get_crs_for_fire(fire_num)
        if csv_crs is None:
            print(f"  ⚠ No CRS found in CSV for fire {fire_num}. Falling back to raster CRS: {raster_crs}")
            export_crs = raster_crs
        else:
            export_crs = csv_crs
            if export_crs != str(raster_crs):
                print(f"  ⚠ Raster CRS ({raster_crs}) differs from CSV CRS ({csv_crs}). Reprojecting classification...")
                classification, transform = reproject_classification(
                    classification, transform, raster_crs, csv_crs
                )
                print(f"  ✅ Reprojected to {csv_crs}")
            else:
                print(f"  ✅ Raster CRS matches CSV CRS, no reprojection needed.")

        # Export results with the chosen CRS and transform
        print("💾 Exporting results...")
        raster_path, vector_path = export_classification_results(
            classification, transform, export_crs, display_fire_id, CLASSIFIED_DIR
        )

        # Create report with RGB visualization
        report_path, stats_path = create_classification_report(classification, band_info, display_fire_id, CLASSIFIED_DIR)

        # Save classifier
        classifier_path = os.path.join(CLASSIFIED_DIR, f"{display_fire_id}_random_forest_classifier.pkl")
        with open(classifier_path, 'wb') as f:
            pickle.dump(rf_classifier, f)
        print(f"    ✅ Classifier saved: {classifier_path}")

        # Display only the classification report
        display_classification_report(report_path, display_fire_id)

        return {
            'fire_id': display_fire_id,
            'fire_num': fire_num,
            'filename': image_file,
            'classifier': rf_classifier,
            'raster_path': raster_path,
            'vector_path': vector_path,
            'report_path': report_path,
            'stats_path': stats_path,
            'classifier_path': classifier_path,
            'training_samples': len(training_points),
            'urban_mask_used': urban_mask is not None,
            'export_crs': str(export_crs)
        }

    except Exception as e:
        print(f"❌ Error processing fire {fire_num}: {e}")
        import traceback
        traceback.print_exc()
        raise  # Re-raise the exception to stop execution

def main():
    """
    (EN) Main workflow: discovers all fires, processes them sequentially, and generates a summary.
    (BG) Основен работен поток: открива всички пожари, обработва ги последователно и генерира обобщение.
    """
    print("🚀 Starting Random Forest Land Cover Classification")
    print(f"📁 Source directory: {PREVIEW_DIR}")
    print(f"📊 Output directory: {CLASSIFIED_DIR}")
    print(f"📚 Polygon directory: {POLYGON_DIR}")
    print(f"🏙️  Urban mask directory: {URBAN_MASK_DIR}")
    print("="*80)

    # Check directories
    if not os.path.exists(PREVIEW_DIR):
        print(f"❌ Source directory not found: {PREVIEW_DIR}")
        sys.exit(1)

    if not os.path.exists(POLYGON_DIR):
        print(f"❌ Polygon directory not found: {POLYGON_DIR}")
        sys.exit(1)

    # Check urban mask directory
    if os.path.exists(URBAN_MASK_DIR):
        print(f"✓ Urban mask directory found: {URBAN_MASK_DIR}")
        urban_files = [f for f in os.listdir(URBAN_MASK_DIR) if f.endswith('.gpkg')]
        if urban_files:
            print(f"  Found {len(urban_files)} urban mask files")
        else:
            print(f"  ⚠ No urban mask files found in directory")
    else:
        print(f"⚠ Urban mask directory not found: {URBAN_MASK_DIR}")
        print("  Will proceed without urban masks")

    # Find maximum fire number
    max_fire_num = 0
    for file in os.listdir(PREVIEW_DIR):
        if file.endswith('.tif') or file.endswith('.tiff'):
            fire_num = extract_fire_number_from_filename(file)
            if fire_num and fire_num > max_fire_num:
                max_fire_num = fire_num

    if max_fire_num == 0:
        print("❌ No fire image files found")
        sys.exit(1)

    print(f"📊 Found fire images up to fire {max_fire_num}")

    # Process fires in order from 1 to max_fire_num
    results = []
    processed_count = 0

    try:
        for fire_num in range(1, max_fire_num + 1):
            print(f"\n{'='*80}")
            print(f"PROCESSING FIRE {fire_num:02d} / {max_fire_num}")
            print(f"{'='*80}")

            result = process_fire_by_number(fire_num)

            if result:
                results.append(result)
                processed_count += 1
                print(f"\n✅ Successfully processed fire {fire_num:02d}")
            else:
                print(f"\n❌❌❌ CRITICAL ERROR: Failed to process fire {fire_num:02d}")
                print("❌❌❌ Stopping execution as requested.")
                sys.exit(1)

    except Exception as e:
        print(f"\n❌❌❌ UNEXPECTED ERROR: {e}")
        print("❌❌❌ Stopping execution.")
        sys.exit(1)

    # Generate summary report
    if results:
        print(f"\n{'='*80}")
        print("📊 CLASSIFICATION SUMMARY")
        print(f"{'='*80}")

        summary_data = []
        for result in results:
            # Load statistics
            if os.path.exists(result['stats_path']):
                stats_df = pd.read_csv(result['stats_path'])
                total_area = stats_df['area_ha'].sum()

                summary_data.append({
                    'Fire ID': result['fire_id'],
                    'Fire #': result['fire_num'],
                    'Training Samples': result['training_samples'],
                    'Urban Mask Used': 'Yes' if result.get('urban_mask_used') else 'No',
                    'Export CRS': result.get('export_crs', 'N/A'),
                    'Classes': len(stats_df),
                    'Total Area (ha)': f"{total_area:.1f}",
                    'Raster': os.path.basename(result['raster_path']),
                    'Report': os.path.basename(result['report_path'])
                })

        if summary_data:
            summary_df = pd.DataFrame(summary_data)
            print("\n📋 Processing Summary:")
            print(summary_df.to_string(index=False))

            # Save summary
            summary_path = os.path.join(CLASSIFIED_DIR, "processing_summary.csv")
            summary_df.to_csv(summary_path, index=False)
            print(f"\n💾 Summary saved: {summary_path}")

            # Count fires with urban masks
            urban_mask_count = sum(1 for r in results if r.get('urban_mask_used'))
            print(f"\n🏙️  Urban masks applied to {urban_mask_count} out of {len(results)} fires")

            print(f"\n✅ Successfully processed {processed_count}/{max_fire_num} fires")

    print(f"\n🎯 Processing complete!")
    print(f"📁 All results saved in: {CLASSIFIED_DIR}")
    print("="*80)

if __name__ == "__main__":
    main()